In [ ]:
# ====== Load filtered segmentation & aggregation result (independent script) ======
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import scanpy as sc 
from shapely.geometry import Point, box

# Auto get project root directory for cross-platform running
ROOT_DIR = os.path.dirname(os.path.abspath(__file__))

# Unified path configuration (modify here only for custom environment)
class Args:
    data_from = os.path.join(ROOT_DIR, "HD_data_halved")
args = Args()

# Load pre-saved filtered h5ad file with relative path
h5ad_file_path = os.path.join(args.data_from, "count_area_filtered_adata_p75.h5ad")
count_area_filtered_adata = sc.read_h5ad(filename=h5ad_file_path)

# Copy raw filtered data for subsequent analysis
final_adata = count_area_filtered_adata.copy()
print(f"Successfully loaded filtered AnnData from: {h5ad_file_path}")

In [ ]:
# Load filtered nucleus segmentation results
gdf_path = os.path.join(args.data_from, "gdf_right_half.parquet")
gdf = gpd.read_parquet(gdf_path)

# Calculate centroid coordinates for each nucleus
gdf["centroid_x"] = gdf["geometry"].centroid.x
gdf["centroid_y"] = gdf["geometry"].centroid.y

# Keep only nuclei that passed QC filtering
gdf_filtered = gdf[gdf["id"].isin(final_adata.obs["id"])].set_index("id")

In [ ]:
# Load cell type deconvolution results and filter valid labels
deconv_path = os.path.join(args.data_from, "DeconvolutionResults_P2CRC.csv")
deconv_df = pd.read_csv(deconv_path)
# Remove rows with missing primary deconvolution cell type labels
deconv_df = deconv_df.dropna(subset=["DeconvolutionLabel1"])

In [ ]:
# Load and filter spatial coordinate information of tissue bins
bin8_path = os.path.join(args.data_from, "tissue_positions.parquet")
bin8_tissue_pos = pd.read_parquet(bin8_path)

# Keep only spots inside tissue region
bin8_tissue_pos = bin8_tissue_pos[bin8_tissue_pos["in_tissue"] == 1]
# Filter spots according to cropped image boundary
bin8_tissue_pos = bin8_tissue_pos[bin8_tissue_pos['pxl_col_in_fullres'] < 53968]
# Set barcode as index for subsequent matching
bin8_tissue_pos = bin8_tissue_pos.set_index("barcode")

In [ ]:
# Merge spatial coordinates with deconvolution cell-type annotations
bin8_annot = pd.merge(
    bin8_tissue_pos[["pxl_col_in_fullres", "pxl_row_in_fullres"]],
    deconv_df.set_index("barcode"),
    left_index=True,
    right_index=True,
    how="inner"
)
# Print number of valid matched 8 μm bins
print(f"===== Valid matched 8 μm bins: {bin8_annot.shape[0]} =====")

In [ ]:
# Create GeoDataFrame for single cells with centroid coordinates
single_cell_geo = gpd.GeoDataFrame(
    final_adata.obs,
    geometry=[Point(gdf_filtered.loc[id, "centroid_x"], gdf_filtered.loc[id, "centroid_y"]) 
              for id in final_adata.obs["id"]],
    crs=gdf.crs
)

In [ ]:
# ====== Unit Conversion & Spatial Matching with 8μm Bins ======
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

# Load spatial scale factors from relative path
scalefactors_path = os.path.join(args.data_from, "spatial", "scalefactors_json.json")
with open(scalefactors_path, "r") as f:
    scalefactors = json.load(f)

# Calculate spatial conversion parameters
microns_per_pixel = scalefactors["microns_per_pixel"] 
bin_size_um = scalefactors["bin_size_um"]  

bin_physical_side = bin_size_um  
bin_pixel_side = bin_physical_side / microns_per_pixel 
bin_pixel_half_side = bin_pixel_side / 2  

# Print unit conversion results
print(f"===== Unit Conversion Results =====")
print(f"1 pixel = {microns_per_pixel:.6f} μm")
print(f"Pixel side length of 8μm bin = {bin_pixel_side:.2f} pixels")
print(f"Pixel boundary of 8μm bin = center ± {bin_pixel_half_side:.2f} pixels")

# Calculate bin boundaries in pixel coordinates
bin8_annot["x_min_pixel"] = bin8_annot["pxl_col_in_fullres"] - bin_pixel_half_side
bin8_annot["x_max_pixel"] = bin8_annot["pxl_col_in_fullres"] + bin_pixel_half_side
bin8_annot["y_min_pixel"] = bin8_annot["pxl_row_in_fullres"] - bin_pixel_half_side
bin8_annot["y_max_pixel"] = bin8_annot["pxl_row_in_fullres"] + bin_pixel_half_side

# Create polygon geometry for each 8μm bin
bin8_annot["geometry"] = bin8_annot.apply(
    lambda row: box(row["x_min_pixel"], row["y_min_pixel"], row["x_max_pixel"], row["y_max_pixel"]),
    axis=1
)
bin8_gdf = gpd.GeoDataFrame(bin8_annot, crs=gdf.crs)  

# Spatial join: match single cells to 8μm bins
spatial_join = gpd.sjoin(
    single_cell_geo, 
    bin8_gdf[["geometry", "DeconvolutionClass", "DeconvolutionLabel1", "DeconvolutionLabel2"]],
    how="left",
    predicate="within"
)

# Calculate matching statistics
spatial_join["matched_success"] = ~spatial_join["DeconvolutionLabel1"].isna()
total_cells = len(spatial_join)
matched_cells = spatial_join["matched_success"].sum()
match_rate = matched_cells / total_cells * 100

# Print matching results
print(f"\n===== Matching Results (Corrected Units) =====")
print(f"Total cells: {total_cells}")
print(f"Successfully matched cells: {matched_cells}")
print(f"Matching rate: {match_rate:.2f}%")

In [ ]:
def assign_single_cell_type(row):
    deconv_class = row["DeconvolutionClass"]
    label1 = row["DeconvolutionLabel1"]
    label2 = row["DeconvolutionLabel2"]
    
    if pd.isna(deconv_class):
        return "Unknown"
    elif deconv_class == "singlet":
        return label1
    elif deconv_class.startswith("doublet"):
        return f"Mixed({label1}+{label2})"
    elif deconv_class == "reject":
        return "Unknown"
    else:
        return "Unknown"

In [ ]:
spatial_join["single_cell_type"] = spatial_join.apply(assign_single_cell_type, axis=1)

In [ ]:
# Deduplicate spatial join results by cell ID (keep first valid match)
spatial_join_dedup = spatial_join.groupby("id").apply(
    lambda x: x[x["matched_success"]].iloc[0] if x["matched_success"].any() else x.iloc[0]
).reset_index(drop=True)

# Keep only valid cells from the filtered AnnData object
valid_cell_ids = final_adata.obs["id"].tolist()
spatial_join_dedup = spatial_join_dedup[spatial_join_dedup["id"].isin(valid_cell_ids)]

# Reorder to match the original cell order in AnnData
spatial_join_dedup = spatial_join_dedup.set_index("id").loc[valid_cell_ids].reset_index()

# Print validation information
print(f"\nNumber of rows after deduplication: {len(spatial_join_dedup)}")
print(f"Consistent with final_adata rows: {len(spatial_join_dedup) == len(final_adata.obs)}")

# Prepare cell type annotation for merging
cell_type_annot = spatial_join_dedup[["id", "single_cell_type"]].set_index("id")

# Merge annotations into AnnData object
final_adata.obs = final_adata.obs.join(cell_type_annot, how="left")

# Fill unannotated cells with "Unknown"
final_adata.obs["single_cell_type"] = final_adata.obs["single_cell_type"].fillna("Unknown")

# Print summary of annotation results
print(f"\nColumns in final_adata.obs after merging: {final_adata.obs.columns.tolist()}")
print(f"Cell type distribution (Top 5):")
print(final_adata.obs["single_cell_type"].value_counts().head(10))

In [ ]:
filter_mask = (final_adata.obs["single_cell_type"] != "Unknown") & (~final_adata.obs["single_cell_type"].str.contains("Mixed\(", na=False))
final_adata_pure = final_adata[filter_mask].copy()

In [ ]:
true_indices = filter_mask[filter_mask].index 
true_position_indices = [filter_mask.index.get_loc(cid) for cid in true_indices]

In [ ]:
# ====== Cell Communication Neighbor Pair Construction ======
import numpy as np
import pickle
import gzip
import pandas as pd
import scanpy as sc
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import euclidean_distances
from collections import defaultdict, Counter
import random
import os

# Auto-detect project root directory
ROOT_DIR = os.path.dirname(os.path.abspath(__file__))

# Unified path configuration (relative paths only)
class Args:
    data_from = os.path.join(ROOT_DIR, "HD_data_halved")
    data_to = os.path.join(ROOT_DIR, "NEST", "metadata1")
args = Args()

# File paths using relative structure
h5ad_path = os.path.join(args.data_from, "count_area_filtered_adata_p75_1.h5ad")
coordinate_barcode_path = os.path.join(args.data_to, "Visium_HD_Human_Colon_Cancer_square_002um_outputs_coordinate_barcode_1")
adjacency_records_path = os.path.join(args.data_from, "input_graph_2", "Visium_HD_Human_Colon_Cancer_square_002um_outputs", "Visium_HD_Human_Colon_Cancer_square_002um_outputs_adjacency_records")
output_path = os.path.join(args.data_from, "combined_communication_pairs.csv")

# Parameter settings
k_nn = 50
spot_diameter = 37.04
neighborhood_threshold = spot_diameter * 4

# true_index_set is defined in previous code
true_index_set = set(true_position_indices)

# Load coordinate and barcode data
with gzip.open(coordinate_barcode_path, 'rb') as fp:
    coordinates_raw, cell_barcode = pickle.load(fp)
print(f"Coordinate data shape: {coordinates_raw.shape} (expected 24068 rows)")

# Load adjacency and communication records
with gzip.open(adjacency_records_path, 'rb') as fp:
    row_col, edge_weight, lig_rec, total_num_cell = pickle.load(fp)
print(f"Total raw communication edges: {len(row_col)}")

# Compute Euclidean distance matrix in 6 chunks to save memory
n_cell = len(coordinates_raw)
distance_matrix = np.zeros((n_cell, n_cell))
print("Start computing Euclidean distance matrix (6 parts)...")
for partition_id in range(1, 7):
    start_j = (n_cell * (partition_id - 1)) // 6
    end_j = (n_cell * partition_id) // 6
    distance_matrix[:, start_j:end_j] = euclidean_distances(
        coordinates_raw, coordinates_raw[start_j:end_j]
    )
    print(f"Distance matrix progress: Part {partition_id} ({start_j} to {end_j})")

# Compute normalized neighbor weight matrix
dist_X = np.zeros_like(distance_matrix)
print("Start computing neighbor weight matrix dist_X...")
for j in range(n_cell):
    if j not in true_index_set:
        continue
    
    dist_j = distance_matrix[:, j]
    max_val = np.max(dist_j)
    min_val = np.min(dist_j)
    if max_val - min_val == 0:
        dist_X[:, j] = 0
    else:
        dist_X[:, j] = 1 - (dist_j - min_val) / (max_val - min_val)
    
    # Retain top-k neighbors only
    top_k_indices = np.argsort(dist_X[:, j])[-k_nn:]
    dist_X[:, j] = np.where(np.arange(n_cell)[:, None] == top_k_indices, dist_X[:, j], 0)

# Extract valid undirected neighbor pairs
neighbor_pairs = set()
print("Extracting valid neighbor pairs...")
for i in range(n_cell):
    if i not in true_index_set:
        continue
    for j in range(n_cell):
        if j not in true_index_set:
            continue
        if dist_X[i, j] != 0:
            pair = tuple(sorted((i, j)))
            neighbor_pairs.add(pair)

print(f"Total valid neighbor pairs (undirected): {len(neighbor_pairs)}")

# Extract valid communication pairs
communication_pairs = set()
for (i, j) in row_col:
    if i in true_index_set and j in true_index_set:
        pair = tuple(sorted((i, j)))
        communication_pairs.add(pair)

comm_count = len(communication_pairs)
print(f"Total valid communication pairs: {comm_count}")

# Get neighbor pairs WITHOUT communication
no_comm_pairs = neighbor_pairs - communication_pairs
no_comm_count = len(no_comm_pairs)
no_comm_list = list(no_comm_pairs)
print(f"Neighbor pairs without communication: {no_comm_count}")

# Balance positive and negative samples
target_count = comm_count
random.seed(42)
if no_comm_count >= target_count:
    selected_no_comm = random.sample(no_comm_list, target_count)
    print(f"Selected {target_count} non-communication pairs (balanced)")
else:
    selected_no_comm = no_comm_list
    print(f"Warning: Only {no_comm_count} non-communication pairs available. Using all.")

# Build balanced dataset
comm_df = pd.DataFrame(communication_pairs, columns=["cell_index_1", "cell_index_2"])
comm_df["has_communication"] = True

no_comm_df = pd.DataFrame(selected_no_comm, columns=["cell_index_1", "cell_index_2"])
no_comm_df["has_communication"] = False

combined_df = pd.concat([comm_df, no_comm_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

# Save final dataset
combined_df.to_csv(output_path, index=False)

# Validation summary
print("\n===== Final Validation =====")
print(f"Total samples: {len(combined_df)}")
print(f"Positive (communication): {len(combined_df[combined_df['has_communication']])}")
print(f"Negative (no communication): {len(combined_df[~combined_df['has_communication']])}")
ratio = len(combined_df[combined_df['has_communication']]) / len(combined_df[~combined_df['has_communication']])
print(f"Balance ratio: {ratio:.2f} (ideal = 1.0)")

# Random sample check
print("\nRandom 5-sample validation:")
sample = combined_df.sample(5, random_state=42)
for _, row in sample.iterrows():
    i, j = row["cell_index_1"], row["cell_index_2"]
    has_comm = row["has_communication"]
    is_neighbor = (dist_X[i, j] != 0) or (dist_X[j, i] != 0)
    real_comm = (tuple(sorted((i, j))) in communication_pairs)
    status = "PASS" if has_comm == real_comm else "FAIL"
    print(f"- Pair ({i},{j}): has_comm={has_comm} | is_neighbor={is_neighbor} | real={real_comm} | {status}")

print(f"\nOutput saved to: {output_path}")